In [1]:
import sys
print(sys.executable)


h:\projects\ai_based\Agent-Factory\.venv\Scripts\python.exe


In [10]:
from fastapi import FastAPI, HTTPException, Header, Depends, Request
from pydantic import BaseModel
from dotenv import load_dotenv
import os

import requests
from typing import List, Dict, Optional
from src.utils.llmp_utils import llmp_call
from src.agents.kb_agent import KBAgent

class GenerateRequest(BaseModel):
    model: str
    system_prompt: str = ''
    prompt: str
    format: Optional[dict] = None
    image: Optional[str] = None
    tools: Optional[List[Dict]] = None
    src: str = None
    temperature: float = 0.5
    
class Agent0:
    
    def __init__(self, tools_desc, model, agents):
        
        
        from dotenv import load_dotenv
        from sentence_transformers import CrossEncoder
        
        load_dotenv()
        
        
        self.src = 'agent_0'
        self.model = model
        self.llmp_url = os.getenv("LLMP_URL")
        self.llmp_password = os.getenv("LLMP_PASSWORD")
        self.tools_desc = tools_desc
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

        self.knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics"'
              }
        print("Initializing Agents!")
        for agent in agents:
            if agent == 'kb_agent':
                self.kb_agent = KBAgent(self.knowledge_bases_desc,self.model)
        
    def llmp_call(self, prompt, system_prompt, model):
        """ 
        Call the LLMP API to generate a response
        All related to the call is processed here
        """
        
        headers = {
        "Content-Type": "application/json",
        "Authorization": self.llmp_password
    }
        
        # Construct request payload
        request_data = GenerateRequest(
        model=model,
        system_prompt=system_prompt,
        prompt=prompt,
        tools=None,
        src=self.src)
        
        payload = request_data.model_dump(exclude_none=True)

        try:
            response = requests.post(self.llmp_url, headers=headers, json=payload)
            response.raise_for_status()  # Raise an error for bad responses (4xx, 5xx)
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            return None
        
    def agent_0_response(self, user_prompt, temperature=0.5):
        """ 
        Encompasses logic behind tool decision making
        Calls the llmp_call method to generate a response
        """
        
        system_prompt = "You are an assistant. Your only goal is to provide me the name of the tool I would need to get the answer to the prompt and nothing else."
        
        tools_description = "\n ".join([f"{key}: {value}" for key, value in self.tools_desc.items()])
        prompt = f"{user_prompt}\nwhich of the following tools would you use?\n {tools_description}"
        
        llmp_response = llmp_call(prompt, system_prompt, self.model, temperature,src = 'Agent 0')['message']['content']
        
        results = self.cross_encoder.predict([[llmp_response, tool] for tool in self.tools_desc.keys()])
        
        tool_scores = dict(zip(self.tools_desc.keys(), results))
        best_tool = max(tool_scores, key=tool_scores.get)
        
        
        return best_tool,user_prompt
        
        
    def agent_0_chat(self, user_prompt):
            """
            Logic behind tool activation.
            Sends to agent_0_response for tool decision.
            Activates tool.

            Args:
                user_prompt (str)
            """
            
            agent_0_response = self.agent_0_response(user_prompt)
            
            selected_tool = agent_0_response[0]
            print(f"Passing to: \n{selected_tool} !")
            
            if selected_tool == 'image_generator':
                
                from src.image_generator import ImageGeneratorAgent
                img_gen = ImageGeneratorAgent()
                img_gen.generate(user_prompt)
                
            if selected_tool == 'ingestion_pipeline':
                
                from src.pipelines.ingestion_pipeline import ingest_pipeline
                ingest_pipeline()
            
            if selected_tool == 'Knowledge Base Query Agent':                
                
                user_prompt_rag = user_prompt.strip('given my documents')
                self.kb_agent.kb_agent_chat(user_prompt_rag)
                
                
                
                
        

In [11]:
agent_0_tools_desc = {'conversational_agent':'a conversational assistant, designed to conduct simple conversations with users',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "llama3.2:latest",['kb_agent'])

# knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
#               'mathematics_kb':'a knowledge base with information related to mathematics"'
#               }


# kb_agent = KBAgent(knowledge_bases_desc, "llama3.2:latest")



In [ ]:
user_prompt = "what is the use of integrals in series convergence test?"
response = kb_agent.kb_agent_chat(user_prompt)

In [12]:
user_prompt = "given my documents, what is the use of integrals in series convergence test?"
response = agent_0.agent_0_chat(user_prompt)

Passing to: 
Knowledge Base Query Agent !
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
The final answer is: To determine whether a series converges or diverges.
📚 References:

📄 **Calculus. Stewart.pdf**
   📑 Pages: 21, 111, 113, 117, 126, 127



In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from src.agent_0 import Agent0
agent_0_tools_desc = {'conversational_agent':'a conversational assistant, designed to conduct simple conversations with users',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "llama3.2:latest",['kb_agent'])



Initializing Agents!
Agents are ready for your use!


In [3]:
user_prompt = "given my documents, what is the definition of an integral?"
response = agent_0.agent_0_chat(user_prompt)

Passing to: 
Knowledge Base Query Agent !
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
The definition of an integral is a mathematical operation that calculates the area under a curve or the accumulation of a quantity over a defined interval. It is denoted by the symbol ∫ and represents the process of finding the total amount of change in a function over a specified range.

In essence, integration is the reverse process of differentiation. While differentiation finds the rate of change of a function, integration finds the total amount of change or accumulation of that function over a given interval.

For example, if we have a function f(x) and we want to find the area under its curve from x = a to x = b, we would write:

∫[a, b] f(x) dx

This represents the integral of f(x) with respect to x, evaluated over the interval [a, b]. The result of this integration is th

## Step 1: Understand the problem statement
The problem asks us to prove that if a function f(x) is continuous and growing (i.e., its derivative f'(x) exists and is positive), then it must pass through the origin (0,0).

## Step 2: Recall the definition of continuity
A function f(x) is said to be continuous at x=a if the following conditions are satisfied:
- The function is defined at x=a.
- The limit of the function as x approaches a exists.
- The limit of the function as x approaches a is equal to the value of the function at x=a.

## Step 3: Recall the definition of a growing function
A function f(x) is said to be growing if its derivative f'(x) exists and is positive for all x in the domain of the function.

## Step 4: Use the Mean Value Theorem (MVT)
The MVT states that if a function f(x) is continuous on the closed interval [a,b] and differentiable on the open interval (a,b), then there exists a point c in (a,b) such that:
f'(c) = (f(b) - f(a)) / (b-a)

## Step 5: Apply the MVT to the function
Let's apply the MVT to the function f(x) on the interval [0,1]. Since f(x) is continuous and growing, we know that f'(x) exists and is positive for all x in the domain of the function.

## Step 6: Use the fact that f(0) = 0
Since f(x) passes through the origin (0,0), we know that f(0) = 0.

## Step 7: Apply the MVT to find a point c where f(c) = 1/2
Using the MVT, we can find a point c in (0,1) such that:
f'(c) = (f(1) - f(0)) / (1-0)
Since f(x) is growing, we know that f'(x) > 0 for all x. Therefore, we have:
f'(c) > 0
(f(1) - 0) / 1 > 0
f(1) > 0

## Step 8: Use the fact that f(x) is continuous and growing to conclude that f(c) = 1/2
Since f(x) is continuous and growing, we know that there exists a point c in (0,1) such that:
f'(c) = (f(1) - f(c)) / (1-c)
Substituting f(1) > 0 and f'(c) > 0, we get:
(f(1) - f(c)) / (1-c) > 0
Since f(x) is continuous, we know that f(c) = 1/2.

The final answer is: $\boxed{\frac{1}{2}}$
📚 References:

📄 **Calculus. Stewart.pdf**
   📑 Pages: 82, 83, 85, 87, 88, 89, 101